https://medium.com/gitconnected/pydanticai-enterprise-grade-ai-development-made-simple-0e4c395f596c

In [1]:
!pip install -qq pydantic-ai pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.8/222.8 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.8/210.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.6/271.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is t

In [22]:
from openai import AsyncAzureOpenAI
from google.colab import userdata

from IPython.display import clear_output
from pprint import pprint

import nest_asyncio

from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel

from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

from dataclasses import dataclass

from pydantic import BaseModel, Field
from pydantic import ValidationError
from pydantic_ai import Agent, RunContext

from IPython.display import clear_output
import json

from bson.objectid import ObjectId
from IPython.display import Markdown, display

import numpy as np
import time

In [3]:
async_azure_client = AsyncAzureOpenAI(api_key=userdata.get('AZURE_API_KEY'), azure_endpoint=userdata.get('AZURE_BASE_URL'), api_version=userdata.get('AZURE_API_VERSION'))

In [4]:
nest_asyncio.apply()

In [5]:
model = OpenAIModel('gpt-4o-2', openai_client=async_azure_client)

In [ ]:
mongo_client = MongoClient(userdata.get("MONGODB_URI"), server_api=ServerApi('1'))
mongo_client.admin.command('ping')
print("Pinged your deployment. You successfully connected to MongoDB!")

Pinged your deployment. You successfully connected to MongoDB!


In [24]:
cws_db = mongo_client['CWS']
cv_data = cws_db['cv_data']
# cv_data.find_one({'candidate_id': 'TEST7000001'}, projection={'filename':1, 'name':1,'email':1})

In [25]:
job_data = cws_db['job_description_data']
# job_data.find_one(ObjectId('672d466137ff38b3b48366a2'), projection={'metadata':1})

In [26]:
class MongoConn:
    """This is a class to request data from MongoDB
    """

    cws_db = mongo_client['CWS']
    cv_data = cws_db['cv_data']

    @classmethod
    async def candidate_name(cls, *, candidate_id: str) -> str | None:
        result = cls.cv_data.find_one({'candidate_id': candidate_id}, projection={'name': 1})
        if result:
            return result['name']

    @classmethod
    async def candidate_raw_resume(cls, *, candidate_id: str) -> str:
        result = cls.cv_data.find_one({'candidate_id': candidate_id}, projection={'data': 1})
        if result:
            return result['data']
        else:
            raise ValueError('Customer not found')

In [27]:
@dataclass
class SupportDependencies:
    candidate_id: str
    db: MongoConn


class SupportResult(BaseModel):
    profession_name: str = Field(description='Name of the proffesion of the candidate')
    it_proffesional: bool = Field(description="Whether candidate is related to Information Technological topic")
    hard_skill: list[str] = Field(description='List of hard skills of the candidate')
    soft_skill: list[str] = Field(description='List of soft skills of the candidate')


candidate_agent = Agent(
    model=model,
    deps_type=SupportDependencies,
    result_type=SupportResult,
    system_prompt=(
        'You are a support agent to retrieve info about candidates and its resume'
    ),
)


@candidate_agent.system_prompt
async def add_customer_name(ctx: RunContext[SupportDependencies]) -> str:
    customer_name = await ctx.deps.db.candidate_name(candidate_id=ctx.deps.candidate_id)
    return f"The candidate's name is {customer_name!r}"


@candidate_agent.tool
async def candidate_resume_info(
    ctx: RunContext[SupportDependencies]
) -> str:
    """Returns the candidate's information include their name."""
    return await ctx.deps.db.candidate_raw_resume(
        candidate_id=ctx.deps.candidate_id,
    )

In [28]:
deps = SupportDependencies(candidate_id='TEST7000001', db=MongoConn())
result = candidate_agent.run_sync('Tell me their information', deps=deps)
pprint(result.data.model_dump())

{'hard_skill': ['Programming: Java, Python, C++, SQL',
                'Development tools: Eclipse, Visual Studio, Git'],
 'it_proffesional': True,
 'profession_name': 'Computer Engineer',
 'soft_skill': ['Communication',
                'Teamwork',
                'Project management',
                'Technical problem solving',
                'Clear and concise presentation of complex ideas']}


In [33]:
deps = SupportDependencies(candidate_id='TEST7000001', db=MongoConn())
async with candidate_agent.run_stream('Tell me their information', deps=deps) as response:
    async for streamed_text in response.stream_structured():
        clear_output(wait=True)
        support_result = streamed_text[0].parts[0].args
        pprint(data)
        time.sleep(1)

clear_output()
pprint(json.loads(support_result))

{'hard_skill': ['Java',
                'Python',
                'C++',
                'SQL',
                'Eclipse',
                'Visual Studio',
                'Git'],
 'it_proffesional': True,
 'profession_name': 'Software Engineer',
 'soft_skill': ['Project management',
                'Technical problem solving',
                'Communication',
                'Teamwork']}


In [51]:
async with candidate_agent.run_stream('Tell me their information', deps=deps) as result:
    async for message, last in result.stream_structured(debounce_by=0.1):
        # print('message:',message)
        try:
            profile = await result.validate_structured_result(
                message,
                # allow_partial=False,
                allow_partial=not last,
            )
            # time.sleep(1)
        except ValidationError:
            continue
        # print('last', last)
        pprint(profile)

SupportResult(profession_name='Software Engineer', it_proffesional=True, hard_skill=['Java', 'Python', 'C++', 'SQL', 'Eclipse', 'Visual Studio', 'Git'], soft_skill=['Communication', 'Teamwork', 'Problem-solving'])
SupportResult(profession_name='Software Engineer', it_proffesional=True, hard_skill=['Java', 'Python', 'C++', 'SQL', 'Eclipse', 'Visual Studio', 'Git'], soft_skill=['Communication', 'Teamwork', 'Problem-solving', 'Project management'])
SupportResult(profession_name='Software Engineer', it_proffesional=True, hard_skill=['Java', 'Python', 'C++', 'SQL', 'Eclipse', 'Visual Studio', 'Git'], soft_skill=['Communication', 'Teamwork', 'Problem-solving', 'Project management'])


In [ ]:
# deps = SupportDependencies(candidate_id='TEST7000001', db=MongoConn())
# async with candidate_agent.run_stream('Tell me the information.be concise in one paragraph', deps=deps) as response:
#     async for streamed_text in response.stream_text():
#         clear_output(wait=True)
#         pprint(streamed_text)


In [ ]:
class MongoConn:
    """This is a class to request data from MongoDB
    """

    cws_db = mongo_client['CWS']
    cv_data = cws_db['cv_data']
    job_data = cws_db['job_description_data']

    @classmethod
    async def candidate_info(cls, *, candidate_id: str) -> str | None:
        result = cls.cv_data.find_one({'candidate_id': candidate_id}, projection={'name': 1, 'experiences': 1, 'certifications': 1, 'summary': 1, })
        if result:
            return str(result)

    @classmethod
    async def job_description_info(cls, *, object_id: str) -> str | None:
        result = cls.job_data.find_one(ObjectId(object_id), projection={'metadata': 1})
        if result:
            return str(result)

@dataclass
class SupportDependencies:
    candidate_id: str
    job_object_id: str
    db: MongoConn

mongo_agent = Agent(
    model=model,
    deps_type=SupportDependencies,
    result_type=str,
    system_prompt='You are a support agent to retrieve info about candidate matches with a job description'
)


@mongo_agent.system_prompt
async def add_customer_info(ctx: RunContext[SupportDependencies]) -> str:
    candidate_info = await ctx.deps.db.candidate_info(candidate_id=ctx.deps.candidate_id)
    return f"The candidate's info is {candidate_info!r}"


@mongo_agent.tool
async def job_description_info(
    ctx: RunContext[SupportDependencies]
) -> str:
    """Returns the job description's information."""
    return await ctx.deps.db.job_description_info(
        object_id=ctx.deps.job_object_id,
    )

In [ ]:
deps = SupportDependencies(candidate_id='TEST7000001', job_object_id='672d466137ff38b3b48366a2', db=MongoConn())
async with mongo_agent.run_stream('Tell me if the candidate match/fit with the job', deps=deps) as response:
    async for streamed_text in response.stream_text():
        clear_output(wait=True)
        display(Markdown(streamed_text))

Based on the job description for a "Network Engineer" and the candidate's profile:

### Education Requirements:
- **Job Requirement**: Bachelor's Degree in Computer Science, Information Technology, or a related field.
- **Candidate's Qualification**: Matches the job requirement as the candidate likely has a computer engineering background.

### Experience Requirements:
- **Job Requirement**: 5+ years of network engineering experience and specific experience in network management, analysis, troubleshooting, and data center management.
- **Candidate's Experience**:
  - Over 5 years of experience in software development, with expertise in programming, project management, and technical problem solving.
  - Experience with SQL databases and Python for automation, but no specific experience in network engineering is mentioned.

### Skills:
- **Required Skills**: Active Secret Clearance, ability to live or relocate locally (Radford, VA), network management, troubleshooting, data center management, and security management strategies.
- **Candidate's Skills**: Strong in software development, Python, SQL databases, but lacks explicit network engineering skills and no mention of security clearance or location flexibility.

### Soft Skills:
- **Job Requirement**: Training ability, interpersonal and teamwork skills, analytical and problem-solving skills, communication abilities.
- **Candidate's Skills**: Excellent communicator, effective teamwork skills, and capable of presenting complex ideas. Matches well with required soft skills.

### Conclusion:
The candidate, John Dupont, shows strong skills in software development and has some pertinent soft skills that align well with the requirements for a Network Engineer. However, the experience and specific technical skills in network engineering, along with security clearance and relocation requirements, are not fully met based on the provided information. He might need to gain more specific network engineering experience to be a perfect fit for this role.

In [ ]:
cws_db = mongo_client['CANDIDATE_JOB']
cv_data = cws_db['CANDIDATE_TEMP2']
pipeline = [
    {
        '$vectorSearch': {
            'queryVector': [0.9] * 384,
            'path': 'embedding_384',
            'index': 'candidate_semantic_search_index_embedding_384',
            'numCandidates': 1000,
            'limit': 10,
        }
    },
    {
        '$match': {
            'summary': {
                '$exists': True,
                '$ne': None,
                '$ne': ''
            },
            'educations': {
                '$exists': True,
                '$ne': None,
                '$ne': []
            },
            'experiences': {
                '$exists': True,
                '$ne': None,
                '$ne': []
            },
            'hard_skills': {
                '$exists': True,
                '$ne': None,
                '$ne': []
            },
            'soft_skills': {
                '$exists': True,
                '$ne': None,
                '$ne': []
            },
            # 'certifications': {
            #     '$exists': True,
            #     '$ne': None,
            #     '$ne': []
            # }
        }
    },
    {
        '$project': {
            # 'embedding_384':0,
            # 'data':0,
            '_id':0,
            'summary':1
        }
    }
]

results = list(cv_data.aggregate(pipeline))
len(results)

3

In [ ]:
results

[{'summary': 'Mariraj Padagalingam is an experienced electrical foreman with a strong background in electrical construction works under MEP projects. He has a diploma in EEE and MEP design, and has worked in various roles including electrical supervisor. He is skilled in electrical power DB and power cable termination, electrical construction wiring, and troubleshooting electrical systems. He is a hard worker, team player, and committed to deadlines.'},
 {'summary': "Jomar B. Herrera is an accounting professional with over 10 years of experience in the accounting sector. He has worked in various companies, gaining broad experience in bookkeeping, tax, accounting, and audit. He has a positive 'can do' attitude and has been involved in implementing systems, developing process improvements, and aligning processes to new accounting systems. His biggest accomplishment was his role as a Business Process Expert in Maersk Group, where he supervised a team of over 40 people."},
 {'summary': 'Ra

In [ ]:
# openai_response = await async_azure_client.embeddings.create(
#     input=["The food was delicious and the waiter...", "The weather is pleasant today"],
#     model=userdata.get('AZURE_EMBEDDING_NAME'),
# )
# openai_response.data[1].embedding

In [ ]:
class MongoConn:
    """This is a class to request data from MongoDB
    """

    cws_db = mongo_client['CANDIDATE_JOB']
    cv_data = cws_db['CANDIDATE_TEMP2']
    job_data = cws_db['JOB_DESCRIPTION']
    pipeline = [
            {
                '$vectorSearch': {
                    'queryVector': [],
                    'path': 'embedding_384',
                    'index': 'candidate_semantic_search_index_embedding_384',
                    'numCandidates': 100,
                    'limit': 10
                }
            },
            {
                '$match': { # returns candidates' resume when they exist educations, expierences and certifications
                    'summary': {'$exists': True, '$ne': None, '$ne': ""},
                    'educations': {'$exists': True, '$ne': None, '$ne': []},
                    'experiences': {'$exists': True, '$ne': None, '$ne': []},
                    'certifications': {'$exists': True, '$ne': None, '$ne': []}
                }
            },
            {
                '$project': { '_id':0, 'summary':1 }
            }
        ]


    @staticmethod
    def normalize_l2(x:list[float]) -> list[float]:
        x = np.array(x)
        if x.ndim == 1:
            norm = np.linalg.norm(x)
            if norm == 0:
                return x.tolist()
            return (x / norm).tolist()
        else:
            norm = np.linalg.norm(x,2,axis=1, keepdims=True)
            return np.where(norm==0, x, x/norm).tolist()

    @classmethod
    async def retrieve_candidates(self, *, job_metadata: str) -> str | None:
        job_metadata_embedding = await async_azure_client.embeddings.create(
            input=[job_metadata],
            model=userdata.get('AZURE_EMBEDDING_NAME'),
        )
        embedding = job_metadata_embedding.data[0].embedding
        self.pipeline[0]['$vectorSearch']['queryVector'] = self.normalize_l2(embedding[:384])
        results = list(self.cv_data.aggregate(self.pipeline))
        if results:
            return str(results)

    @classmethod
    async def job_description_info(self, *, object_id: str) -> str | None:
        result = self.job_data.find_one(ObjectId(object_id), projection={'position_summary': 1, 'required_qualifications':1})
        if result:
            return str(result)

    @classmethod
    async def job_description_name(self, *, object_id: str) -> str | None:
        result = self.job_data.find_one(ObjectId(object_id), projection={'job_name': 1})
        if result:
            return result['job_name']

@dataclass
class SupportDependencies:
    # candidate_id: str
    job_object_id: str
    num_candidates: int
    db: MongoConn

mongo_agent = Agent(
    model=model,
    deps_type=SupportDependencies,
    result_type=str,
    system_prompt='You are a support agent to retrieve info about candidates matches with a job descriptions'
)


@mongo_agent.system_prompt
async def add_job_description_name(ctx: RunContext[SupportDependencies]) -> str:
    print('Calling... add_job_description_name')
    job_name = await ctx.deps.db.job_description_name(object_id=ctx.deps.job_object_id)
    return f"The job's name is {job_name!r}, and only use the {ctx.deps.num_candidates} most fit candidates."


@mongo_agent.tool
async def job_description_info(
    ctx: RunContext[SupportDependencies]
) -> str:
    """Returns the job description's information."""
    print('Calling... job_description_info tool')
    return await ctx.deps.db.job_description_info(object_id=ctx.deps.job_object_id)

@mongo_agent.tool
async def retrieve_candidates(
    ctx: RunContext[SupportDependencies], job_description_info: str
) -> str:
    """Returns the possible candidates info to match with a job"""
    print('Calling... retrieve_candidates tool')
    return await ctx.deps.db.retrieve_candidates(job_metadata=job_description_info)

In [ ]:
deps = SupportDependencies(job_object_id='676053f062db4613ac84d6ed', num_candidates=3, db=MongoConn())
async with mongo_agent.run_stream("""
Generate a short table with the most fit candidates for the job description, then tell me why match or not with its required qualifications
""", deps=deps) as response:
    async for streamed_text in response.stream_text():
        clear_output(wait=True)
        display(Markdown(streamed_text))

### Candidates Table

| Name             | Summary                                                                 | Match with Required Qualifications                                          |
|------------------|-------------------------------------------------------------------------|-----------------------------------------------------------------------------|
| Tamilselvan R    | System Administrator with over 9 years of experience in IT/Manufacturing|  Matches with networking experience, lacks mention of security clearance.   |
| Deepak Chandra   | Technology-driven professional with over 8 years in IT Administration   |  Matches with both networking experience and proficiency in installation, but lacks mention of security clearance and degree. |
| Gaurav P Nirmal  | ITIL certified with over 9 years of experience in Knowledge Management  |  Matches partly on experience with IT services, but lacks specific network engineering focus and security clearance. |

### Match Analysis

1. **Tamilselvan R**:
   - *Strengths*: Extensive experience in system and network administration aligns well with the job's focus on network engineering. 
   - *Weaknesses*: Security clearance status is not mentioned, and there is no confirmation of a relevant degree or ability to relocate to Radford, VA.

2. **Deepak Chandra**:
   - *Strengths*: Solid experience in networking, system maintenance, and IT administration. 
   - *Weaknesses*: Missing information on security clearance and educational background; lacks details about relocation capability or current location.

3. **Gaurav P Nirmal**:
   - *Strengths*: Broad ITIL expertise and project management skills.
   - *Weaknesses*: Lacks specific network engineering focus, detail on security clearance, and potential relocation capabilities. Emphasis is more on IT services than engineering.

In [ ]:
deps = SupportDependencies(job_object_id='676055ba62db4613ac850237', num_candidates=3, db=MongoConn())
async with mongo_agent.run_stream("""
Generate a short table with the most fit candidates for the job description, then tell me why match or not with its required qualifications
""", deps=deps) as response:
    async for streamed_text in response.stream_text():
        clear_output(wait=True)
        display(Markdown(streamed_text))

Here's a summary of the most fit candidates for the Java Developer position based on the job description:

| Candidate               | Summary                                                                                                                                                                                                                                                                                                           | Matches & Discrepancies                                                                                                                                                                 |
|-------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Visionary Java Architect| An experienced architect with 12+ years in designing enterprise-level applications. Holds a Master's in Software Engineering.                                                                                                                                                                                     | **Matches:** Extensive experience with Java and enterprise applications.<br>**Discrepancies:** Details on travel ability and experience with Spring not specified.                       |
| Hitesh Sagar            | Experienced in sales, marketing management, and more, seeking a challenging job to leverage skills.                                                                                                                                                                                                               | **Matches:** Passionate about customer strategy, marketing, and platform development.<br>**Discrepancies:** Lacks Java experience; focus is more generalized management and marketing.   |
| Gaurav P Nirmal         | ITIL certified with over 9 years in ITIL Based Service Delivery, Knowledge Management, Project Management. Currently a Knowledge Manager at Unisys Global Services India.                                                                                                                                         | **Matches:** Problem-solving skills, leadership experience.<br>**Discrepancies:** No Java or Spring experience, primarily focused on ITIL services and management.                    |

### Summary
1. **Visionary Java Architect** is the most aligned technically due to experience in Java but may lack specifics in Spring experience and travel requirements.
2. **Hitesh Sagar** has experience in customer strategy areas but diverges from the core technical aspects required, notably Java.
3. **Gaurav P Nirmal** also does not meet the technical requirements around Java, despite strong project management credentials. 

These evaluations are based on the available information, but further details may be required to fully evaluate each candidate's qualifications relative to the requirements of the job description.

# Emulate MultiDatabases

In [ ]:
class HardSkillConnect:
    """This is a class to request hard skills of candidates
    """

    cws_db = mongo_client['CANDIDATE_JOB']
    cv_data = cws_db['CANDIDATE_TEMP2']

    @classmethod
    async def candidate_info(self, *, security_id: str) -> str | None:
        result = self.cv_data.find_one(ObjectId(security_id), projection={'hard_skills': 1})
        if result:
            return str(result)

In [ ]:
class SoftSkillConnect:
    """This is a class to request soft skills of candidates
    """

    cws_db = mongo_client['CANDIDATE_JOB']
    cv_data = cws_db['CANDIDATE_TEMP2']

    @classmethod
    async def candidate_info(self, *, security_id: str) -> str | None:
        result = self.cv_data.find_one(ObjectId(security_id), projection={'soft_skills': 1})
        if result:
            return str(result)

In [ ]:
class SummaryConnect:
    """This is a class to request summaries of candidates
    """

    cws_db = mongo_client['CANDIDATE_JOB']
    cv_data = cws_db['CANDIDATE_TEMP2']

    @classmethod
    async def candidate_info(self, *, security_id: str) -> str | None:
        result = self.cv_data.find_one(ObjectId(security_id), projection={'summary': 1})
        if result:
            return str(result)

In [ ]:
class CandidateMetrics:
    """This is a class to request Candidates metrics
    """

    cws_db = mongo_client['CANDIDATE_JOB']
    cv_data = cws_db['CANDIDATE_TEMP2']

    @classmethod
    async def candidate_info(self, *, security_id: str) -> str | None:
        result = self.cv_data.find_one(ObjectId(security_id), projection={'hard_skills': 1,'soft_skills': 1})
        if result:
            return f"The candidate has {len(result['hard_skills'])} hard skills and {len(result['soft_skills'])} soft skills."

In [ ]:
@dataclass
class SupportDependencies:
    hs_db: HardSkillConnect
    ss_db: SoftSkillConnect
    summary_db: SummaryConnect
    metrics_db: CandidateMetrics


mongo_agent = Agent(
    model=model,
    deps_type=SupportDependencies,
    result_type=str,
    system_prompt='You are a support agent to retrieve info about candidates using a security id, always refer their name'
)


# @mongo_agent.system_prompt
# async def add_job_description_name(ctx: RunContext[SupportDependencies]) -> str:
#     job_name = await ctx.deps.db.job_description_name(object_id=ctx.deps.job_object_id)
#     return f"The job's name is {job_name!r}, and only use the {ctx.deps.num_candidates} most fit candidates."


@mongo_agent.tool
async def candidate_hard_skills(ctx: RunContext[SupportDependencies], security_id:str) -> str:
    """Returns the candidates'hard skills."""
    return await ctx.deps.hs_db.candidate_info(security_id=security_id)

@mongo_agent.tool
async def candidate_soft_skills(ctx: RunContext[SupportDependencies], security_id:str) -> str:
    """Returns the candidates'soft skills."""
    return await ctx.deps.ss_db.candidate_info(security_id=security_id)

@mongo_agent.tool
async def candidate_summary(ctx: RunContext[SupportDependencies], security_id:str) -> str:
    """Returns the candidates'summary."""
    return await ctx.deps.summary_db.candidate_info(security_id=security_id)

In [ ]:
deps = SupportDependencies(hs_db=HardSkillConnect(), ss_db=SoftSkillConnect(), summary_db=SummaryConnect(), metrics_db=None)
async with mongo_agent.run_stream("""Retrieve all data about the candidate with the security id: 673e012eeb476b40e0c2d2b6""", deps=deps) as response:
    async for streamed_text in response.stream_text():
        clear_output(wait=True)
        display(Markdown(streamed_text))

Here is the information for Mike Kisasati Wanaswa:

**Summary:**
Mike Kisasati Wanaswa is an experienced Fixed Data Network Technician with a strong background in telecommunication and information technology. He has worked with various companies, including Ben's Electronics Services Ltd, and has expertise in installation, maintenance, and support of various technologies. He holds a diploma in Computer Engineering and has completed several professional training courses.

**Hard Skills:**
- **Network Management:** Advanced proficiency
- **Data Recovery:** Advanced proficiency
- **Fiber Optics Splicing:** Advanced proficiency
- **CCTV Installation:** Advanced proficiency
- **Wi-Fi Setup:** Advanced proficiency

**Soft Skills:**
- **End User Training:** Advanced proficiency
- **Technical Support:** Advanced proficiency

Please let me know if you need more details or further assistance!

In [ ]:
async with mongo_agent.run_stream("""Retrieve all data about the candidate with the security id: 673e012eeb476b40e0c2d2b6""", deps=deps) as response:
    async for streamed_text in response.stream_text():
        clear_output(wait=True)
        display(Markdown(streamed_text)) # accumulated string

The candidate with the security ID 673e012eeb476b40e0c2d2b6 is Mike Kisasati Wanaswa. Here's the detailed information:

### Summary:
Mike Kisasati Wanaswa is an experienced Fixed Data Network Technician with a strong background in telecommunication and information technology. He has worked with various companies, including Ben’s Electronics Services Ltd, and has expertise in installation, maintenance, and support of various technologies. He holds a diploma in Computer Engineering and has completed several professional training courses.

### Hard Skills:
1. **Network Management**: Proficiency Level: Advanced
2. **Data Recovery**: Proficiency Level: Advanced
3. **Fiber Optics Splicing**: Proficiency Level: Advanced
4. **CCTV Installation**: Proficiency Level: Advanced
5. **Wi-Fi Setup**: Proficiency Level: Advanced

### Soft Skills:
1. **End User Training**: Proficiency Level: Advanced
2. **Technical Support**: Proficiency Level: Advanced

If you need further details or have any other queries about Mike Kisasati Wanaswa, feel free to ask!

In [ ]:
@mongo_agent.tool
async def candidate_metrics(ctx: RunContext[SupportDependencies], security_id:str) -> str:
    """Returns the candidates' metrics."""
    return await ctx.deps.metrics_db.candidate_info(security_id=security_id)

deps = SupportDependencies(hs_db=HardSkillConnect(), ss_db=SoftSkillConnect(), summary_db=SummaryConnect(), metrics_db=CandidateMetrics())
async with mongo_agent.run_stream("""Retrieve all data about the candidate with the security id: 673e012eeb476b40e0c2d2b6""", deps=deps) as response:
    async for streamed_text in response.stream_text():
        clear_output(wait=True)
        display(Markdown(streamed_text))

The candidate, Mike Kisasati Wanaswa, is an experienced Fixed Data Network Technician with a strong background in telecommunication and information technology. Here's a breakdown of his skills and qualifications:

### Summary
- Mike has worked with various companies, including Ben’s Electronics Services Ltd.
- He has expertise in installation, maintenance, and support of various technologies.
- He holds a diploma in Computer Engineering and has completed several professional training courses.

### Hard Skills
- **Network Management**: Advanced proficiency
- **Data Recovery**: Advanced proficiency
- **Fiber Optics Splicing**: Advanced proficiency
- **CCTV Installation**: Advanced proficiency
- **Wi-Fi Setup**: Advanced proficiency

### Soft Skills
- **End User Training**: Advanced proficiency
- **Technical Support**: Advanced proficiency

### Metrics
- Mike has a total of 5 hard skills and 2 soft skills.

If you need additional information about Mike's qualifications or experience, feel free to ask!

In [ ]:
deps = SupportDependencies(hs_db=HardSkillConnect(), ss_db=SoftSkillConnect(), summary_db=SummaryConnect(), metrics_db=CandidateMetrics())
async with mongo_agent.run_stream("""Retrieve all metrics about the candidate with the security id: 673e012eeb476b40e0c2d2b6""", deps=deps) as response:
    async for streamed_text in response.stream_text():
        clear_output(wait=True)
        display(Markdown(streamed_text))

The candidate associated with the security ID 673e012eeb476b40e0c2d2b6 has 5 hard skills and 2 soft skills. If you need further details about these specific skills or any other information, please let me know!

In [ ]:
deps = SupportDependencies(hs_db=None, ss_db=None, summary_db=None, metrics_db=CandidateMetrics())
async with mongo_agent.run_stream("""Retrieve all data about the candidate with the security id: 673e012eeb476b40e0c2d2b6""", deps=deps) as response:
    async for streamed_text in response.stream_text():
        clear_output(wait=True)
        display(Markdown(streamed_text))

AttributeError: 'NoneType' object has no attribute 'candidate_info'

In [ ]:
deps = SupportDependencies(hs_db=None, ss_db=None, summary_db=None, metrics_db=CandidateMetrics())
async with mongo_agent.run_stream("""Retrieve only the metrics about the candidate with the security id: 673e012eeb476b40e0c2d2b6""", deps=deps) as response:
    async for streamed_text in response.stream_text():
        clear_output(wait=True)
        display(Markdown(streamed_text))

The candidate with the security id 673e012eeb476b40e0c2d2b6 has 5 hard skills and 2 soft skills.

# TRYING MULTIPLE TIMES DIFFERENT QUERIES

In [ ]:
query_skill = 'java'

# MongoDB query to find users whose hard_skills contain a similar skill to 'vava'
query = {
    'hard_skills': {
        '$elemMatch': {
            'skill_name': {
                '$regex': query_skill,  # Search for skill_name similar to the input
                '$options': 'i'         # Case-insensitive search
            }
        }
    }
}

cws_db = mongo_client['CANDIDATE_JOB']
cv_data = cws_db['CANDIDATE_TEMP2']
matching_users = list(cv_data.find(query, {'summary':1}).limit(10))
len(matching_users)

10

In [ ]:
type(cv_data.find(query, {'summary':1}))

pymongo.synchronous.cursor.Cursor

In [ ]:
class QueryConnection:
    """This is a class to request mongo query in MongoDB CANDIDATE collection
    """

    cws_db = mongo_client['CANDIDATE_JOB']
    cv_data = cws_db['CANDIDATE_TEMP2']
    tries = 0

    ## @classmethod
    async def candidates_summaries(self, *, mongo_query: str, limit:int) -> str | None:
        self.tries += 1
        print('This is the try number:', self.tries)
        try:
            if self.tries == 1:
                return 'Use php instead of ava'
            elif self.tries == 2:
                return 'Use javascript instead of php'
            elif self.tries == 3:
                return 'The mongo query returns empty, try other query'
            else:
                result = list(self.cv_data.find(json.loads(mongo_query), {'summary':1}).limit(limit))
        except Exception as e:
            return f"fix the mongo query due to the following error: {str(e)}"
        if not result:
            return 'The mongo query returns empty, try other query'
        else:
            return str(result)

    @classmethod
    async def document_structure(cls) -> str | None:
        return """
        {
            ...
            'hard_skills': [
                {
                    'skill_name': '...',
                    'years_of_experience': '...',
                    'skill_name': '...',
                }
            ]
            ...
        }
        """

In [ ]:
@dataclass
class SupportDependencies:
    db: QueryConnection


mongo_agent = Agent(
    model=model,
    deps_type=SupportDependencies,
    result_type=str,
    system_prompt='''
    You are a support agent to retrieve info about candidates based on user input,
     when a tool returns empty or fails, try other query according the indications, retries 8 max
    '''
)


@mongo_agent.system_prompt
async def add_job_description_name(ctx: RunContext[SupportDependencies]) -> str:
    print('Calling... add_job_description_name')
    doc = await ctx.deps.db.document_structure()
    return f"The structure of the mongo document is: {doc}"


@mongo_agent.tool(retries=6)
async def candidate_info(ctx: RunContext[SupportDependencies], mongo_query:str, limit:int) -> str:
    """Returns the candidates'info based on mongo query.
    Args:
        mongo_query: str mongo query to find candidates all atributes must be in double quotes, use regex
        limit: int number of candidates to return
    """
    print(f'Calling... candidate_info tool with {mongo_query=}')
    return await ctx.deps.db.candidates_summaries(mongo_query=mongo_query,limit=limit)

In [ ]:
deps = SupportDependencies(db=QueryConnection())
async with mongo_agent.run_stream("""Retrieve candidates who knows vava, list three candidates of them""", deps=deps) as response:
    async for streamed_text in response.stream_text():
        # clear_output(wait=True)
        display(Markdown(streamed_text))

Calling... add_job_description_name
Calling... candidate_info tool with mongo_query='{"hard_skills.skill_name": {"$regex": "vava", "$options": "i"}}'
This is the try number: 1
Calling... candidate_info tool with mongo_query='{"hard_skills.skill_name": {"$regex": "php", "$options": "i"}}'
This is the try number: 2
Calling... candidate_info tool with mongo_query='{"hard_skills.skill_name": {"$regex": "javascript", "$options": "i"}}'
This is the try number: 3
Calling... candidate_info tool with mongo_query='{"hard_skills.skill_name": {"$regex": "java", "$options": "i"}}'
This is the try number: 4


Here are three candidates who know Java:

1. **Amitendra G

Here are three candidates who know Java:

1. **Amitendra Gaurav**: 
   - Seeking an entry-level position in Networking, web-design, or other fields where

Here are three candidates who know Java:

1. **Amitendra Gaurav**: 
   - Seeking an entry-level position in Networking, web-design, or other fields where he can apply his programming knowledge.
   - Completed a B.E from RGTU in 2014 with a CG

Here are three candidates who know Java:

1. **Amitendra Gaurav**: 
   - Seeking an entry-level position in Networking, web-design, or other fields where he can apply his programming knowledge.
   - Completed a B.E from RGTU in 2014 with a CGPA of 6.47.
   - Skills: Microsoft Office, Internet, Operating

...

Here are three candidates who know Java:

1. **Amitendra Gaurav**: 
   - Seeking an entry-level position in Networking, web-design, or other fields where he can apply his programming knowledge.
   - Completed a B.E from RGTU in 2014 with a CGPA of 6.47.
   - Skills: Microsoft Office, Internet, Operating Systems, CCNA, HTML, CSS, JavaScript, and Basic C.

2. **Vishal Saini**: 
   - Fresher in Mechanical Engineering, currently in the 7th semester of a B.Tech course at Skyline Institute of Engineering & Technology.
   - Basic knowledge of C & JAVA.
   - Skills: Windows 7 and Windows 8 operating systems

Here are three candidates who know Java:

1. **Amitendra Gaurav**: 
   - Seeking an entry-level position in Networking, web-design, or other fields where he can apply his programming knowledge.
   - Completed a B.E from RGTU in 2014 with a CGPA of 6.47.
   - Skills: Microsoft Office, Internet, Operating Systems, CCNA, HTML, CSS, JavaScript, and Basic C.

2. **Vishal Saini**: 
   - Fresher in Mechanical Engineering, currently in the 7th semester of a B.Tech course at Skyline Institute of Engineering & Technology.
   - Basic knowledge of C & JAVA.
   - Skills: Windows 7 and Windows 8 operating systems, AutoCAD, and MS Office software.
   - Summer training at BHEL Haridwar focused on steam turbine manufacturing

Here are three candidates who know Java:

1. **Amitendra Gaurav**: 
   - Seeking an entry-level position in Networking, web-design, or other fields where he can apply his programming knowledge.
   - Completed a B.E from RGTU in 2014 with a CGPA of 6.47.
   - Skills: Microsoft Office, Internet, Operating Systems, CCNA, HTML, CSS, JavaScript, and Basic C.

2. **Vishal Saini**: 
   - Fresher in Mechanical Engineering, currently in the 7th semester of a B.Tech course at Skyline Institute of Engineering & Technology.
   - Basic knowledge of C & JAVA.
   - Skills: Windows 7 and Windows 8 operating systems, AutoCAD, and MS Office software.
   - Summer training at BHEL Haridwar focused on steam turbine manufacturing.

3. **Amitendra Gaurav**: 
   - Another

Here are three candidates who know Java:

1. **Amitendra Gaurav**: 
   - Seeking an entry-level position in Networking, web-design, or other fields where he can apply his programming knowledge.
   - Completed a B.E from RGTU in 2014 with a CGPA of 6.47.
   - Skills: Microsoft Office, Internet, Operating Systems, CCNA, HTML, CSS, JavaScript, and Basic C.

2. **Vishal Saini**: 
   - Fresher in Mechanical Engineering, currently in the 7th semester of a B.Tech course at Skyline Institute of Engineering & Technology.
   - Basic knowledge of C & JAVA.
   - Skills: Windows 7 and Windows 8 operating systems, AutoCAD, and MS Office software.
   - Summer training at BHEL Haridwar focused on steam turbine manufacturing.

3. **Amitendra Gaurav**: 
   - Another summary of the same candidate, potentially indicating repeated entry in the database.
   - Same qualifications and skills as above. 

These candidates

Here are three candidates who know Java:

1. **Amitendra Gaurav**: 
   - Seeking an entry-level position in Networking, web-design, or other fields where he can apply his programming knowledge.
   - Completed a B.E from RGTU in 2014 with a CGPA of 6.47.
   - Skills: Microsoft Office, Internet, Operating Systems, CCNA, HTML, CSS, JavaScript, and Basic C.

2. **Vishal Saini**: 
   - Fresher in Mechanical Engineering, currently in the 7th semester of a B.Tech course at Skyline Institute of Engineering & Technology.
   - Basic knowledge of C & JAVA.
   - Skills: Windows 7 and Windows 8 operating systems, AutoCAD, and MS Office software.
   - Summer training at BHEL Haridwar focused on steam turbine manufacturing.

3. **Amitendra Gaurav**: 
   - Another summary of the same candidate, potentially indicating repeated entry in the database.
   - Same qualifications and skills as above. 

These candidates have diverse backgrounds with relevant Java knowledge.

In [ ]:
mongo_agent.name

'mongo_agent'

# Retries in sqlite

In [ ]:
from abc import ABC, abstractmethod
from enum import Enum
from dataclasses import dataclass
import sqlite3
from functools import cached_property

class SQLiteConnection:
    """A class to handle SQLite database queries and schema retrieval."""

    conn = sqlite3.connect('hrdb.db')

    ## @classmethod
    async def execute_query(self, *, query: str) -> str | None:
        """Executes a given SQL query and returns the result or an error message."""
        # self.tries += 1
        try:
            result = self.conn.cursor().execute(query).fetchall()
        except Exception as e:
            print(f"Error executing query: {str(e)}")
            return f"Error executing query: {str(e)}"

        if not result or result[0][0]==0:
            return f"The query returned no results. Please try a different query, not {query}"
        else:
            return str(result)

    @cached_property
    async def get_database_schema(self) -> str:
        """Retrieves the schema of all tables in the SQLite database."""
        cursor = self.conn.cursor()
        schema_list = []
        schema_query = "SELECT sql FROM sqlite_master WHERE type='table' AND name!='sqlite_sequence';"
        tables = cursor.execute(schema_query).fetchall()

        for table_schema in tables:
            if table_schema[0]:
                schema_list.append(table_schema[0])

        return '\n\n'.join(schema_list)


@dataclass
class DependencySupport:
    db: SQLiteConnection


sqlite_agent = Agent(
    model=model,
    deps_type=DependencySupport,
    result_type=str,
    system_prompt=(
    'You are a support agent designed to answer user inquiries about an SQLite database.'
    'When a tool fails/returns/responses empty try using other query, max 6 tries'
    # 'List country names beforehand to fix user grammar'
    # 'Always At then end mention the full workflow(inputs and outputs) of each tool'
    'For countries, if a tool fails or returns empty, try it again but use short abbrevations'
    # 'For queries in  should be complex to reduce number of calls if its possible'
    )
)


@sqlite_agent.system_prompt
async def fetch_database_schema(ctx: RunContext[DependencySupport]) -> str:
    """Fetches the schema of the SQLite database."""
    print('Executing... fetch_database_schema')
    return f"The database schema is:\n{await ctx.deps.db.get_database_schema()}"


@sqlite_agent.tool(retries=5)
async def employee_queries(ctx: RunContext[DependencySupport], query: str) -> str:
    """Handles user queries related to the SQLite database.

    Args:
        query (str): The SQL query to execute
    """
    print(f'Executing... employee_queries with {query=}')
    return await ctx.deps.db.execute_query(query=query)


In [ ]:
deps = SupportDependencies(db=SQLiteConnection())
async with sqlite_agent.run_stream("""Retrieve employees from unitedstates""", deps=deps, message_history=[]) as response:
    async for streamed_text in response.stream_text():
        # clear_output(wait=True)
        display(Markdown(streamed_text))

Executing... fetch_database_schema
Executing... employee_queries with query="SELECT * FROM employees WHERE country_id = (SELECT country_id FROM countries WHERE country_name = 'United States')"
Executing... employee_queries with query="SELECT * FROM employees WHERE country_id = (SELECT country_id FROM countries WHERE country_name = 'USA')"


Here are the employees from the United States:

1. **Viren V. Timbadiya**
   - Email: viren0312@gmail.com
   - Phone: +1-555-1234
   - Hire Date: 2019

Here are the employees from the United States:

1. **Viren V. Timbadiya**
   - Email: viren0312@gmail.com
   - Phone: +1-555-1234
   - Hire Date: 2019-03-22
   - Job Title: Pharmacy and Marketing Specialist
   - Salary: $70,000
   - Department ID: 2
   - State ID: 1
   - Country ID: 1

2. **Ashutosh

Here are the employees from the United States:

1. **Viren V. Timbadiya**
   - Email: viren0312@gmail.com
   - Phone: +1-555-1234
   - Hire Date: 2019-03-22
   - Job Title: Pharmacy and Marketing Specialist
   - Salary: $70,000
   - Department ID: 2
   - State ID: 1
   - Country ID: 1

2. **Ashutosh Mishra**
   - Email: ashu.mishra166@yahoo.com
   - Phone: +1-555-8765
   - Hire Date: 2019-12-05
   - Job Title: Finance Specialist
   - Salary: $70,

Here are the employees from the United States:

1. **Viren V. Timbadiya**
   - Email: viren0312@gmail.com
   - Phone: +1-555-1234
   - Hire Date: 2019-03-22
   - Job Title: Pharmacy and Marketing Specialist
   - Salary: $70,000
   - Department ID: 2
   - State ID: 1
   - Country ID: 1

2. **Ashutosh Mishra**
   - Email: ashu.mishra166@yahoo.com
   - Phone: +1-555-8765
   - Hire Date: 2019-12-05
   - Job Title: Finance Specialist
   - Salary: $70,000
   - Department ID: 2
   - State ID: 1
   - Country ID: 1

In [ ]:
async with sqlite_agent.run_stream("""Retrieve employees from peru""", deps=deps, message_history=[]) as response:
    async for streamed_text in response.stream_text():
        # clear_output(wait=True)
        display(Markdown(streamed_text))

Executing... fetch_database_schema
Executing... employee_queries with query="SELECT employees.name\nFROM employees\nJOIN countries ON employees.country_id = countries.country_id\nWHERE countries.country_name = 'Peru';"
Executing... employee_queries with query="SELECT e.name, e.email, e.phone_number, e.hire_date, e.job_title, e.salary, d.department_name, s.state_name, c.country_name \nFROM employees e\nJOIN countries c ON e.country_id = c.country_id\nJOIN departments d ON e.department_id = d.department_id\nJOIN states s ON e.state_id = s.state_id\nWHERE c.country_name = 'PE';"
Executing... employee_queries with query="SELECT employees.name\nFROM employees\nJOIN countries ON employees.country_id = countries.country_id\nWHERE countries.country_name = 'PE';"


It appears that there are no employees from Peru stored in the database. Would you like assistance with anything else?

In [ ]:
%%timeit -n1 -r5
r=sqlite_agent.run_sync("""How many employees exist from cand and eeuu""", deps=deps, message_history=[])
print(r.all_messages()[-1].parts[0].content)

Executing... fetch_database_schema
Executing... employee_queries with query="SELECT COUNT(*) FROM employees e JOIN countries c ON e.country_id = c.country_id WHERE c.country_name = 'Canada';"
Executing... employee_queries with query="SELECT COUNT(*) FROM employees e JOIN countries c ON e.country_id = c.country_id WHERE c.country_name = 'United States';"
Executing... employee_queries with query="SELECT COUNT(*) FROM employees e JOIN countries c ON e.country_id = c.country_id WHERE c.country_name = 'USA';"
There are 2 employees from Canada and 2 employees from the USA (United States).
Executing... fetch_database_schema
Executing... employee_queries with query="SELECT COUNT(*) FROM employees JOIN countries ON employees.country_id = countries.country_id WHERE countries.country_name = 'Canada'"
Executing... employee_queries with query="SELECT COUNT(*) FROM employees JOIN countries ON employees.country_id = countries.country_id WHERE countries.country_name = 'United States'"
Executing... emp

In [ ]:
r=sqlite_agent.run_sync("""How many employees exist from cand and eeuu""", deps=deps, message_history=[])
print(r.all_messages()[-1].parts[0].content)

Executing... fetch_database_schema
Executing... employee_queries with query="SELECT COUNT(*) FROM employees e JOIN countries c ON e.country_id = c.country_id WHERE c.country_name = 'Canada'"
Executing... employee_queries with query="SELECT COUNT(*) FROM employees e JOIN countries c ON e.country_id = c.country_id WHERE c.country_name = 'United States'"
Executing... employee_queries with query="SELECT COUNT(*) FROM employees e JOIN countries c ON e.country_id = c.country_id WHERE c.country_name = 'CAN'"
Executing... employee_queries with query="SELECT COUNT(*) FROM employees e JOIN countries c ON e.country_id = c.country_id WHERE c.country_name = 'CA'"
Executing... employee_queries with query="SELECT COUNT(*) FROM employees e JOIN countries c ON e.country_id = c.country_id WHERE c.country_name = 'US'"
Executing... employee_queries with query="SELECT COUNT(*) FROM employees e JOIN countries c ON e.country_id = c.country_id WHERE c.country_name = 'USA'"
There are 2 employees from Canada, a

In [ ]:
%%timeit -n1 -r4
r=sqlite_agent.run_sync("""How many employees exist from cand and eeuu""", deps=deps)
print(r.all_messages()[-1].parts[0].content)

Executing... fetch_database_schema
Executing... employee_queries with query="SELECT COUNT(*) FROM employees e JOIN countries c ON e.country_id = c.country_id WHERE c.country_name = 'Canada'"
Executing... employee_queries with query="SELECT COUNT(*) FROM employees e JOIN countries c ON e.country_id = c.country_id WHERE c.country_name = 'United States' OR c.country_name = 'USA'"
There are 2 employees from Canada and 2 employees from the United States (or USA).
Executing... fetch_database_schema
Executing... employee_queries with query="SELECT COUNT(*) AS count FROM employees e JOIN countries c ON e.country_id = c.country_id WHERE c.country_name = 'Canada'"
Executing... employee_queries with query="SELECT COUNT(*) AS count FROM employees e JOIN countries c ON e.country_id = c.country_id WHERE c.country_name = 'USA'"
There are 2 employees from Canada and 2 employees from the USA.
Executing... fetch_database_schema
Executing... employee_queries with query="SELECT COUNT(*) FROM employees WHE

In [ ]:
# DO NOT WORK ASKING HISTORICAL MESSAGES
# r=sqlite_agent.run_sync("""reply me again""", deps=deps)
# print(r.all_messages()[-1].parts[0].content)

In [ ]:
r=sqlite_agent.run_sync("""How many employees exist from cand and ussaa, first list country names to fix grammar""", deps=deps, message_history=[])
print(r.all_messages()[-1].parts[0].content)

Executing... fetch_database_schema
Executing... employee_queries with query='SELECT country_name FROM countries'
Executing... employee_queries with query="SELECT COUNT(*) FROM employees WHERE country_id IN (SELECT country_id FROM countries WHERE country_name IN ('Cand', 'USSAA'));"
Executing... employee_queries with query='SELECT country_name FROM countries'
Executing... employee_queries with query="SELECT COUNT(*) FROM employees WHERE country_id IN (SELECT country_id FROM countries WHERE country_name IN ('Canada', 'USA'))"
The correct country names are "Canada" and "USA." There are 4 employees from these countries.


In [ ]:
pprint(r.all_messages(),width=160)

[ModelRequest(parts=[SystemPromptPart(content='You are a support agent designed to answer user inquiries about an SQLite database.When a tool '
                                              'fails/returns/responses empty try using other query, max 6 triesFor countries, if a tool fails or returns '
                                              'empty, try it again but use short abbrevations',
                                      part_kind='system-prompt'),
                     SystemPromptPart(content='The database schema is:\n'
                                              'CREATE TABLE countries (\r\n'
                                              '    country_id INTEGER PRIMARY KEY AUTOINCREMENT,\r\n'
                                              '    country_name TEXT NOT NULL\r\n'
                                              ')\n'
                                              '\n'
                                              'CREATE TABLE states (\r\n'
                          

# Adding system prompts

In [ ]:
sqlite_agent = Agent(
    model=model,
    deps_type=DependencySupport,
    result_type=str,
    system_prompt=(
    'You are a support agent designed to answer user inquiries about an SQLite database.'
    'Always try to use complex query to reduce number of calls for tools'
    )
)


@sqlite_agent.system_prompt
async def fetch_database_schema(ctx: RunContext[DependencySupport]) -> str:
    """Fetches the schema of the SQLite database."""
    print('Executing... fetch_database_schema')
    return f"The database schema is:\n{await ctx.deps.db.get_database_schema()}"

@sqlite_agent.system_prompt
async def available_countries(ctx: RunContext[DependencySupport]) -> str:
    """Fetches the available countries in the SQLite database."""
    print('Executing... available_countries')
    return f"The available countries are: {await ctx.deps.db.execute_query(query='SELECT country_name FROM countries')}"


@sqlite_agent.tool
async def employee_queries(ctx: RunContext[DependencySupport], query: str) -> str:
    """Handles user queries related to the SQLite database.

    Args:
        query (str): The SQL query to execute, use available data if the user does not provide correctly
    """
    print(f'Executing... employee_queries with {query=}')
    return await ctx.deps.db.execute_query(query=query)


In [ ]:
r=sqlite_agent.run_sync("""How many employees exist from cand and united stat, in each one""", deps=deps, message_history=[])
print(r.all_messages()[-1].parts[0].content)

Executing... fetch_database_schema
Executing... available_countries
Executing... employee_queries with query="SELECT countries.country_name, COUNT(employees.security_id) FROM employees JOIN countries ON employees.country_id = countries.country_id WHERE countries.country_name IN ('Canada', 'USA') GROUP BY countries.country_name;"
There are 2 employees from Canada and 2 employees from the USA.


In [ ]:
pprint(r.all_messages(),width=160)

[ModelRequest(parts=[SystemPromptPart(content='You are a support agent designed to answer user inquiries about an SQLite database.Always try to use complex '
                                              'query to reduce number of calls for tools',
                                      part_kind='system-prompt'),
                     SystemPromptPart(content='The database schema is:\n'
                                              'CREATE TABLE countries (\r\n'
                                              '    country_id INTEGER PRIMARY KEY AUTOINCREMENT,\r\n'
                                              '    country_name TEXT NOT NULL\r\n'
                                              ')\n'
                                              '\n'
                                              'CREATE TABLE states (\r\n'
                                              '    state_id INTEGER PRIMARY KEY AUTOINCREMENT,\r\n'
                                              '    state_name TEXT NOT N

# Add new data in sqlite

In [ ]:
# from pydantic_ai import Agent, ModelRetry, RunContext

# class SQLiteConnection:
#     """A class to handle SQLite database queries and schema retrieval."""

#     conn = sqlite3.connect('hrdb.db')

#     ## @classmethod
#     async def execute_query(self, *, query: str) -> str | None:
#         """Executes a given SQL query and returns the result or an error message."""
#         # self.tries += 1
#         try:
#             result = self.conn.cursor().execute(query).fetchall()
#         except Exception as e:
#             print(f"Error executing query: {str(e)}")
#             return f"Error executing query: {str(e)}"
#         if not result or result[0][0]==0:
#             print('No results')
#             return f"The query returned no results. Please try a different query, not {query}"
#         else:
#             return str(result)

#     ## @classmethod
#     async def get_database_schema(self) -> str | None:
#         """Retrieves the schema of all tables in the SQLite database."""
#         cursor = self.conn.cursor()
#         schema_list = []
#         schema_query = "SELECT sql FROM sqlite_master WHERE type='table' AND name!='sqlite_sequence';"
#         tables = cursor.execute(schema_query).fetchall()

#         for table_schema in tables:
#             if table_schema[0]:
#                 schema_list.append(table_schema[0])

#         return '\n\n'.join(schema_list)


# ##@dataclass
# class DependencySupport:
#     db: SQLiteConnection

# sqlite_agent = Agent(
#     model=model,
#     deps_type=DependencySupport,
#     result_type=str,
#     system_prompt=(
#     'You are a support agent designed to answer user inquiries about an SQLite database.'
#     'Retry use a tool when different query when it fails or return empty'
#     )
# )


# @sqlite_agent.system_prompt
# async def fetch_database_schema(ctx: RunContext[DependencySupport]) -> str:
#     """Fetches the schema of the SQLite database."""
#     print('Executing... fetch_database_schema')
#     return f"The database schema is:\n{await ctx.deps.db.get_database_schema()}"


# @sqlite_agent.tool(retries=5)
# async def employee_queries(ctx: RunContext[DependencySupport], query: str) -> str:
#     """Handles user queries related to the SQLite database.

#     Args:
#         query (str): The SQL query to execute, use available data if the user does not provide correctly
#     """
#     print(f'Executing... employee_queries with {query=}')
#     return await ctx.deps.db.execute_query(query=query)

# deps = SupportDependencies(db=SQLiteConnection())
# r=sqlite_agent.run_sync("""How many employees exist from united stat/usa""", deps=deps, message_history=[])
# print(r.all_messages()[-1].parts[0].content)

Executing... fetch_database_schema
Executing... employee_queries with query="SELECT COUNT(*) FROM employees e JOIN countries c ON e.country_id = c.country_id JOIN states s ON e.state_id = s.state_id WHERE c.country_name LIKE 'United States' OR c.country_name LIKE 'USA'"
There are 2 employees from the "United States" or "USA" in the database.
